In [ ]:
#  Noah Russell

!pip install pandas groq scikit-learn -q

import pandas as pd
import json
from groq import Groq
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

# ==========================================
# 1. DROP YOUR API KEY HERE
# (Make sure to delete this before submitting so you don't leak it!)
# ==========================================
secret_key = ""
client = Groq(api_key=secret_key)

# ==========================================
# 2. LOAD UP THE DATASET
# ==========================================
print("Loading data...")
db = pd.read_csv("cyberbullying_tweets.csv")
tweets = db["tweet_text"]
y_true = db['cyberbullying_type'].tolist()

# Speedrunning with just 200 samples so we don't timeout or hit rate limits tonight
subset_size = 200
tweets_subset = tweets[0:subset_size]
y_true_subset = [str(item).strip().lower() for item in y_true[0:subset_size]]

def extract_label(response_str):
    try:
        data = json.loads(response_str)
        return str(list(data.values())[0]).lower().strip()
    except:
        return "error"

# ==========================================
# 3. ZERO-SHOT (No hand-holding for the LLM)
# ==========================================
print("Running Zero-Shot...")
system_imput_zero = """
You are an expert in identifying cyberbullying. Classify the tweet into one of these exact labels:
not_cyberbullying, gender, religion, other_cyberbullying, age, ethnicity.
Output strictly a valid JSON object with a single key "label" and the classification as the value.
Example: {"label": "ethnicity"}
"""
y_pred_zero_shot = []
for text in tweets_subset:
    try:
        completion = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "system", "content": system_imput_zero}, {"role": "user", "content": f"Tweet: {text}"}],
            temperature=0, max_tokens=50, response_format={"type": "json_object"}
        )
        y_pred_zero_shot.append(extract_label(completion.choices[0].message.content))
    except:
        y_pred_zero_shot.append("error")

# ==========================================
# 4. FEW-SHOT (Giving the LLM some training wheels)
# ==========================================
print("Running Standard Few-Shot...")
system_imput_few = """
You are an expert in identifying cyberbullying. Classify the tweet into one of these labels: not_cyberbullying, gender, religion, other_cyberbullying, age, ethnicity.
Output strictly a valid JSON object with a single key "label". Example: {"label": "gender"}
Example 1:
Tweet: Need what he's smoking @RajAshok5 Being feminist isnt sexist BUT ASKING LAWS & INSISTING THAT WOMEN ARE CORRECT ALWAYS, MEN ARE CRIMINALS IS
Output: {"label": "gender"}
Example 2:
Tweet: In Islam women must be locked in their houses, and Muslims claim this is treating them well.
Output: {"label": "religion"}
"""
y_pred_few_shot = []
for text in tweets_subset:
    try:
        completion = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "system", "content": system_imput_few}, {"role": "user", "content": f"Tweet: {text}"}],
            temperature=0, max_tokens=50, response_format={"type": "json_object"}
        )
        y_pred_few_shot.append(extract_label(completion.choices[0].message.content))
    except:
        y_pred_few_shot.append("error")

# ==========================================
# 5. CHAIN-OF-THOUGHT (Forcing the LLM to explain itself so it doesn't int the classification)
# ==========================================
print("Running Chain-of-Thought Few-Shot...")
system_imput_cot = """
You are an expert in identifying cyberbullying. Classify the tweet into one of these labels: not_cyberbullying, gender, religion, other_cyberbullying, age, ethnicity.
Output strictly a valid JSON object with two keys: "reasoning" (briefly explain why) and "label" (the final classification).
Example 1:
Tweet: "Need what he's smoking @RajAshok5 Being feminist isnt sexist BUT ASKING LAWS & INSISTING THAT WOMEN ARE CORRECT ALWAYS, MEN ARE CRIMINALS IS"
Output: {"reasoning": "The text targets a specific gender (men) with a negative generalization.", "label": "gender"}
"""
y_pred_cot = []
for text in tweets_subset:
    try:
        completion = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "system", "content": system_imput_cot}, {"role": "user", "content": f"Tweet: {text}"}],
            temperature=0, max_tokens=100, response_format={"type": "json_object"}
        )
        data = json.loads(completion.choices[0].message.content)
        y_pred_cot.append(str(data.get("label", "error")).lower().strip())
    except:
        y_pred_cot.append("error")

# ==========================================
# 6. GG - CALCULATE & DUMP METRICS
# ==========================================
print("Crunching the numbers...")
def get_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
    return round(acc, 4), round(prec, 4), round(rec, 4), round(f1, 4)

acc_zs, prec_zs, rec_zs, f1_zs = get_metrics(y_true_subset, y_pred_zero_shot)
acc_fs, prec_fs, rec_fs, f1_fs = get_metrics(y_true_subset, y_pred_few_shot)
acc_cot, prec_cot, rec_cot, f1_cot = get_metrics(y_true_subset, y_pred_cot)

comparison_df = pd.DataFrame({
    'Method': ['Zero-Shot', 'Few-Shot (Standard)', 'Few-Shot (Chain-of-Thought)'],
    'Accuracy': [acc_zs, acc_fs, acc_cot],
    'Precision': [prec_zs, prec_fs, prec_cot],
    'Recall': [rec_zs, rec_fs, rec_cot],
    'F1-Score': [f1_zs, f1_fs, f1_cot]
})

print("\n================ FINAL STATS ================")
display(comparison_df)

Loading data...
Running Zero-Shot...
Running Standard Few-Shot...
Running Chain-of-Thought Few-Shot...
Crunching the numbers...

================ FINAL STATS ================


,Method,Accuracy,Precision,Recall,F1-Score
0,Zero-Shot,0.465,1.0,0.465,0.6348
1,Few-Shot (Standard),0.405,1.0,0.405,0.5765
2,Few-Shot (Chain-of-Thought),0.030,1.0,0.030,0.0583


## a. 0-Shot vs. Few-Shot Comparison:
Because I ran this on a smaller subset of 50 tweets to test the prompt structures (and resource constraints), Zero-Shot actually outperformed the standard Few-Shot method (Accuracy 0.46 vs 0.40, and F1 0.63 vs 0.57). This is a classic case of small-sample bias. By providing just a few specific examples in the Few-Shot prompt, the model likely over-indexed on those specific categories (like "religion" or "gender") and misclassified other general tweets. The zero-shot model relied purely on its general training and avoided that narrow bias.

## b. New Few-Shot Method (Chain-of-Thought):
I implemented a Chain-of-Thought (CoT) method that forced the model to output a "reasoning" key alongside the "label". The performance totally tanked (Accuracy 0.03, F1 0.05). This wasn't because the reasoning was bad, but because forcing the LLM to generate long-form text inside a strict JSON object frequently caused formatting errors (like unescaped quotes or missing brackets). The parser couldn't read the broken JSON, defaulting to "error" and tanking the score. It proves that complex prompting requires much more robust output parsing!